# Harness Engineering: OpenCode Case Study
## Chapter 4: Evaluation and Improvement: Did the Change Help?

In the first three chapters, you inspected agent actions and checked the results. Now use those checks to identify unfinished work, ask the agent to complete it, and check again.

Our familiar grade program has been repaired. The test passes, and the agent says it has written the report. **Would you accept the work?**

You are the reviewer. Use program checks and the saved actions to decide what is finished and what needs more work. We will inspect a real incomplete delivery, check a complete example, and optionally ask OpenCode to finish the missing report.

**Part 1** checks the work and optionally asks the agent to finish the report. **Part 2** explains the workflow. **Part 3** traces the checks, feedback, and recheck in code.

### Part 1: Check the Work, Then Finish the Report
#### 1. What is still missing?

Start with the saved files and actions from a real OpenCode `1.18.34` run using Ollama `qwen3:8b`. These are historical records from a separate run, not results from your earlier chapters. The setup restores the files into a fresh folder; it does not run a model, install OpenCode, or ask for a key.

Read the original request, the agent's answer, and `review.md`. The other files are available below so you can check the requirements yourself.

In [ ]:
#@title Load the lab and its saved records
from pathlib import Path
import json
import re
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/SleepyLGod/AIST4020-Lab-Tutorials.git"
LAB_RELATIVE = Path("labs/opencode/evaluation")
LAB_DIR = next((p for p in (Path.cwd(), Path.cwd() / LAB_RELATIVE)
                if (p / "evaluation_support.py").is_file()), None)
if LAB_DIR is None:
    course = Path(tempfile.mkdtemp(prefix="aist4020-course-"))
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(course)], check=True)
    LAB_DIR = course / LAB_RELATIVE
sys.path.insert(0, str(LAB_DIR))
import evaluation_support as support
if Path(support.__file__).resolve() != (LAB_DIR / "evaluation_support.py").resolve():
    raise RuntimeError("Restart the runtime to load this lab's helper.")
lab = support.open_lab(version="1.18.34")
incomplete_delivery = lab.historical("incomplete")
print("Original task:", incomplete_delivery.record["task"])
support.show_answer(incomplete_delivery)
support.show_project(incomplete_delivery)

#### 2. How do we check more than the final answer?

The original instructions require a working program and a report with **Test results**, **Changes**, and **Unresolved issues**. You can find these requirements in `README.md` and `AGENTS.md`.

We can run the test, compare files, and find missing report sections with code. Whether the report tells the truth takes another step: compare its claims with the recorded actions.

The function below runs your test separately from the agent. Here, `delivery.before` holds the files **before the original code repair**; `allowed_changes` lists what that task could change. `report_sections()` returns heading names and their content, including subheadings. Below, `REQUIRED_SECTIONS` names the headings we need; `report_gaps()` flags a heading if its content has no letters or numbers. It does not judge whether a claim is true.

In [ ]:
TEST_COMMAND = ["python3", "-m", "unittest", "-v"]
REQUIRED_SECTIONS = ("Test results", "Changes", "Unresolved issues")

def report_gaps(text: str) -> list[str]:
    """Find required sections with no letters or numbers in their content."""
    sections = support.report_sections(text)
    return [name for name in REQUIRED_SECTIONS
            if not re.search(r"[A-Za-z0-9]", sections.get(name.casefold(), ""))]

def check_delivery(delivery: support.Delivery, allowed_changes: set[str]) -> dict:
    """Check the program, report sections, and scope of file changes."""
    test = delivery.check(TEST_COMMAND)
    changed = support.changed_files(delivery.before, support.base.snapshot(delivery.project))
    return {
        "test_passed": test.returncode == 0 and not test.timed_out,
        "test_status": support.base.test_outcome(test),
        "test_output": test.stdout + test.stderr,
        "report_gaps": report_gaps(delivery.read("review.md")),
        "changed_files": changed,
        "comparison_start": delivery.comparison_start,
        "unexpected_changes": sorted(set(changed) - allowed_changes),
    }

incomplete_checks = check_delivery(incomplete_delivery, allowed_changes={"grades.py", "review.md"})
support.show_checks(incomplete_delivery.label, incomplete_checks)
support.show_evidence(incomplete_delivery)

The test passes, but the report has only a title. The saved final answer says the report was written; the file contradicts it. The task is unfinished even though the code repair worked.

The agent did run the test; your independent test is a separate check. An earlier search was refused, but later actions repaired and tested the program. A tool error tells you to check what happened next, not that the task stayed blocked.

#### 3. Would the same check accept a complete delivery?

Here is a separate DeepSeek run that completed the original task, not a follow-up to the incomplete run. Apply the same checks: do they recognize complete work? Later, we will return to the incomplete files and inspect a report follow-up.

Can you find the recorded failing test before the edit and the passing test after it? Does the reported code change match the diff? A heading with words underneath is not enough by itself.

In [ ]:
complete = lab.historical("complete")
complete_checks = check_delivery(complete, allowed_changes={"grades.py", "review.md"})
support.show_checks(complete.label, complete_checks)
support.show_evidence(complete)

#### 4. What should we send back to the agent?

Return to the incomplete delivery. Its program already passes the supplied test. We need the report completed, not another implementation change.

Build feedback from the check you just ran. Before sending it, compare the request below with the files and test output: does it describe the actual omission and leave completed work alone? You can review it even with model calls switched off.

In [ ]:
feedback = f"""Complete the unfinished delivery in this project.
Original task: {incomplete_delivery.record['task']}

These are files saved after an earlier run, not work you performed in this session.
The independent check of the current files reports:
{json.dumps(incomplete_checks, indent=2)}

Current review.md:
{incomplete_delivery.read('review.md')}

Read the project instructions. Complete the missing report sections, inspect the
current code, and run python3 -m unittest -v to verify its current state.
Keep grades.py, the tests, data, and instructions unchanged.
Distinguish the existing implementation from what you do in this session.
Report only checks you actually ran and changes supported by the files.
"""
print(feedback)

Provider and model choices match the earlier chapters, with Ollama selected by default. Here, `ENABLE_MODEL_CALLS = False` lets you study the saved records without calling a model. To try one live follow-up, choose your provider and set `ENABLE_MODEL_CALLS = True`. Keys come from Colab Secrets or hidden input, never from a notebook cell.

Hosted APIs can charge for several model calls within this one task. Use a GPU for Ollama. A live run starts a **new session** from the historical files; it does not resume the original Ollama conversation.

In [ ]:
BACKENDS = {
    "deepseek": dict(base_url="https://api.deepseek.com/v1",
                     model="deepseek-flash", key_env="DEEPSEEK_API_KEY"),
    "nvidia": dict(base_url="https://integrate.api.nvidia.com/v1",
                   model="z-ai/glm-5.3-flash", key_env="NVIDIA_NIM_API_KEY"),
    "ollama": dict(base_url="http://127.0.0.1:11434/v1",
                   model="qwen3.5:9b-q4_K_M", key_env=""),
    "custom": dict(base_url="", model="", key_env=""),
}
backend = "ollama"
ENABLE_MODEL_CALLS = False

# Keep the historical tool permissions. Check separately that the follow-up
# leaves the already repaired program unchanged.
PERMISSIONS = {
    "*": "deny", "read": "allow", "glob": "allow", "grep": "allow",
    "edit": {"*": "deny", "grades.py": "allow", "**/grades.py": "allow",
             "review.md": "allow", "**/review.md": "allow"},
    "bash": {"*": "deny", "python3 -m unittest -v": "allow"},
    "external_directory": "deny",
}
assert PERMISSIONS == incomplete_delivery.record["permissions"]
MAX_REMEDIATIONS = 1
AGENT_STEPS = 12
TASK_TIMEOUT_SECONDS = 300
OPENCODE_SETTINGS = {
    "permission": {"*": "deny"},
    "agent": {"build": {"steps": AGENT_STEPS, "permission": PERMISSIONS}},
}

The next cell restores a separate copy, installs the pinned CLI, and writes `OPENCODE_SETTINGS` alongside the model connection in a configuration file. `OPENCODE_CONFIG` points OpenCode to it; `agent.build.permission` holds this agent's rules. The original rule still permits editing `grades.py`; our request asks the agent to leave it alone, and our file check will verify that it did.

The helper prints the command and working directory. It does not write the report or retry a task. `AGENT_STEPS` asks OpenCode to wrap up after that many model steps; the five-minute timeout stops the process if it is still running.

In [ ]:
live = None
live_result = None
if ENABLE_MODEL_CALLS:
    if not incomplete_checks["test_passed"] or incomplete_checks["unexpected_changes"]:
        raise RuntimeError("Inspect the starting files before attempting this report-only follow-up.")
    if incomplete_checks["report_gaps"]:
        try:
            live, binary, model = lab.prepare_live(
                incomplete_delivery, backend, BACKENDS, settings=OPENCODE_SETTINGS,
                enabled=ENABLE_MODEL_CALLS,
            )
        except (OSError, RuntimeError, ValueError) as error:
            print(f"Live setup could not finish: {type(error).__name__}. No follow-up was run.")
            print(lab.redact(str(error)))
        else:
            command = [str(binary), "run", "--pure", "--format", "json",
                       "--agent", "build", "--model", model, feedback]
            for attempt in range(MAX_REMEDIATIONS):
                live_result = lab.run_once(live, command, timeout=TASK_TIMEOUT_SECONDS)
            lab.collect(live, binary)
    else:
        print("No missing report sections found; no follow-up requested.")
else:
    print("Live calls are off. The saved records remain available below.")

#### 5. Did the change help this delivery?

Inspect the files even if the live task stopped early. This time, `delivery.before` holds the files **just before the report follow-up**, with the program already repaired. Only `review.md` should change. Repeat the checks and compare the report with the recorded actions.

Did the report include a test the agent really ran? Did it distinguish existing code from this session's changes? Did any file change beyond the report?

In [ ]:
live_checks = None
if live_result is not None:
    live_checks = check_delivery(live, allowed_changes={"review.md"})
    support.show_answer(live)
    support.show_evidence(live)
    support.compare(incomplete_checks, live_checks, live)
else:
    print("No live task result to assess. This does not count as a follow-up failure.")

Next, examine the **saved report follow-up**, separately from your result. A new DeepSeek session started with the Ollama run's repaired program and unfinished report. It completed the report without changing the other files. The file comparison starts just before that follow-up. This historical result does not show a higher success rate for the original model.

The table reports the extra task time, model responses, tool requests, and tokens. A response count is not an exact HTTP request count, and the saved cost field is not a provider bill.

In [ ]:
saved = lab.historical("remediated")
saved_checks = check_delivery(saved, allowed_changes={"review.md"})
support.show_answer(saved)
support.show_evidence(saved)
support.compare(incomplete_checks, saved_checks, saved)

#### 6. What if the report is still unfinished?

Judge your live run and the saved example separately. Do not keep retrying until you get the answer you hoped for.

| What you observe | What you can conclude |
| --- | --- |
| Report complete, claims supported, program and other files unchanged | The missing work is complete. Consider whether the extra work was worthwhile. |
| Report still incomplete, or claims contradicted by the record | The check found an omission, but this attempt did not resolve it. |
| Report complete, but tests fail or do not finish, or other files changed | Do not accept it yet. Read the test output and file differences to find out what went wrong. |
| Tool history unavailable | You can inspect the files, but cannot confirm claims about the agent's actions. |
| Provider connection failed before work began | The live attempt was not tested. Use the labelled historical case to finish the analysis. |

Write a short conclusion about the run you inspected: **what was missing, what changed, what you checked before accepting or rejecting it, and what extra work it took.**

### Part 2: What Changed in the Harness?
#### 1. Evaluation starts with the task

**Evaluation** means judging the result against the requirements. Our task had two deliverables: repaired code and a report. Testing only the code left part of the task unchecked.

| Requirement | How we judge it |
| --- | --- |
| Missing scores are excluded from the average | The notebook runs the supplied test, which expects 90 |
| Deliver a report | The notebook flags empty or missing sections; you read whether they explain the work |
| Describe actual work | You compare the report's claims with tool records and file changes |
| Preserve the repaired program | The notebook checks the test and file differences; you inspect any unexpected changes |

If the report says "all tests passed," find the test output that supports it. If that record is missing, leave the claim unconfirmed. This is human evaluation: judging against the task requirements, not just trusting the agent or giving a personal impression.

Checks can be performed by code, another model, or a person. This lab combines code with your judgment; it does not use a model to grade the report. See [Anthropic's guide to agent evaluations](https://www.anthropic.com/research/demystifying-evals-for-ai-agents) for the broader comparison.

#### 2. Feedback gives the agent a specific next job

"Check your work" does not say what was missed. Our feedback includes the passing test, the actual title-only file, and the sections that need content. The agent can focus on the unfinished report.

The feedback does not grant new permissions or supply the completed answer. The agent still has to inspect files, run the check, and write the report itself.

The earlier chapters checked results. Here, those findings also decide whether to ask for one follow-up:

| After the check | Next step |
| --- | --- |
| Delivery complete and claims supported | Accept it |
| A concrete omission is found | Send back what is missing, ask once, then check again |
| Work remains incomplete or evidence is missing | Keep it marked unfinished for review |

OpenCode provides the conversation and action records; our lab adds the delivery checks. You review the findings, send a specific request, and judge the new result. That arrangement is the harness change here, not a change to model weights. OpenCode's [CLI](https://opencode.ai/docs/cli/) supports sending requests and exporting session records.

OpenCode also supports [permission approval](https://opencode.ai/docs/permissions/): asking you whether an action may run. Approval answers "May it do this?" Evaluation answers "Did it meet the task requirements?" This lab uses the existing permission rules, not interactive approval prompts.

#### 3. Improvement includes the cost and the remaining uncertainty

In the saved example, the report became complete and the program stayed unchanged. The table records the extra model work, but not your time reading files and checking claims.

Also consider the interaction: could you see what was missing, give a clear correction, and verify the new report? A useful workflow should help you do those things. We have not measured whether this one saves human time.

We can conclude that **this unfinished report was completed without changing the program**. The follow-up used a new session and a different model. It cannot tell us whether feedback alone caused the improvement, or how often this would work on other tasks.

For a broader comparison, we would need multiple tasks, consistent starting conditions and models, and the same acceptance checks. Here, learn to make a well-supported judgment about one delivery before trying to compare whole systems.

### Part 3: Trace the Check, Feedback, and Recheck

The delivery checker is course code; OpenCode runs the agent and records its actions. Keep those responsibilities separate as you trace the example.

The excerpts use [OpenCode v1.18.34](https://github.com/anomalyco/opencode/tree/v1.18.34). `// ...` omits unrelated source lines. Read `yield*` as "perform this operation and get its result."

#### 1. Check what is missing

In Part 1, `check_delivery(...)` runs the test, checks report sections, and compares files. It reads the actual work, not the final answer.

Once the starting files pass the program and file-change checks, this condition decides whether to ask OpenCode for more work:

```python
if incomplete_checks["report_gaps"]:
    # ...
```

*Excerpt from the live-run cell in Part 1; the body is omitted here.*

An empty list means there are no missing sections to send back. It does not prove the report is truthful: you still compare its claims with the actions. The required headings and gap check are visible in Part 1; only the Markdown parsing lives in [evaluation_support.py](evaluation_support.py).

#### 2. The finding becomes a new user request

In Part 1, you pass `feedback` as the final argument to `opencode run`. It enters through the same CLI code as in Chapter 1; this time the request describes the work our checks found missing:

```ts
const model = pick(args.model)
const result = await client.session.prompt({
  sessionID,
  agent,
  model,
  variant: args.variant,
  parts: [...files, { type: "text", text: message }],
})
```

*Source: [cli/cmd/run.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts)*

`message` is our feedback text; `model` is the provider/model choice. We attach no files here, so `files` is empty. The agent reads the restored project through tools. Because our command supplies no `--session`, it starts a new conversation rather than loading the failed run's history.

`MAX_REMEDIATIONS = 1` allows one follow-up task. OpenCode runs the model/tool loop inside it; the helper records the result and stops it if time runs out, without retrying.

#### 3. Check the result again

As in Chapter 1, OpenCode saves each completed tool's result. Here, those records help us check the report:

```ts
const match = yield* readToolCall(toolCallID)
if (!match || match.part.state.status !== "running") return
yield* session.updatePart({
  ...match.part,
  state: {
    status: "completed",
    input: match.part.state.input,
    output: output.output,
    // ...
  },
})
```

*Source: [session/processor.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts)*

`input` preserves the requested command or file operation; `output` preserves what it returned. A completed test command can still report failed tests, so we inspect the result, not just the word `completed`.

The CLI also reports completed or failed tool actions in JSON:

```ts
if (part.type === "tool" && (part.state.status === "completed" || part.state.status === "error")) {
  if (emit("tool_use", { part })) continue
  // ...
}
```

*Source: [cli/cmd/run.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts)*

The notebook displays these actions. Then you repeat the check, expecting only the report to have changed:

```python
live_checks = check_delivery(live, allowed_changes={"review.md"})
```

*From the live-check cell in Part 1.*

Compare the report with the recorded actions as well. If sections are missing, claims lack evidence, or the program changed, leave the work marked unfinished; do not retry until it looks successful.

The useful change was not a longer instruction to "be careful." It was a place in the workflow where an omission could be found, returned, and checked again.

To keep your own records, download them before deleting the Colab runtime. Review them before sharing. The supplied historical excerpts and their provenance are in [records/README.md](records/README.md).

In [ ]:
DOWNLOAD_RECORDS = False
if DOWNLOAD_RECORDS:
    lab.download_records()